# AI-READI Naive Thresholding Comparison

This notebook constructs pooled-quantile naive thresholds on the fixed AI-READI cohort,
converts each subject's glucose trajectory into TIR-style compositions, and compares
naive thresholding against the settled DE baselines using the same linear-model style
as the existing real-data notebook.

In [1]:
import importlib.util
import subprocess
import sys

from tools.r_tools import ensure_r_packages, setup_r_environment

setup_r_environment()

if importlib.util.find_spec("rpy2") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rpy2"])

ensure_r_packages(["clarkeTest"])

Using R installation at: C:\Program Files\R\R-4.4.1


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from statsmodels.formula.api import ols

from tools.ai_readi_tools import add_tir_metrics, format_response_name, load_ai_readi_cohort
from tools.prediction_experim_functions import clarke_test, pack_lm_model, r_squared

REPO_ROOT = Path.cwd()

DE_THRESHOLD_SETS = {
    "de_k2": [96, 170],
    "de_k4": [90, 128, 172, 232],
}

RESPONSES = ["hdl_c", "log_triglycerides", "AIP"]

## Recreate the current AI-READI analytic cohort

The cohort below should match the fixed 573-subject cohort used in the current real-data comparison workflow.

In [3]:
filtered_data = load_ai_readi_cohort(REPO_ROOT)

print("Number of subjects:", filtered_data.shape[0])
print("HbA1c range:", filtered_data["hba1c"].min(), "--", filtered_data["hba1c"].max())

filtered_data[["id", "hdl_c", "log_triglycerides", "AIP", "hba1c"]].head()

Number of subjects: 573
HbA1c range: 4.5 -- 7.4


,id,hdl_c,log_triglycerides,AIP,hba1c
0,1001,92.0,4.418841,-0.102948,5.7
1,1002,41.0,5.030438,1.316866,5.6
2,1006,91.0,4.276666,-0.234193,5.4
3,1008,62.0,4.488636,0.361502,6.0
4,1011,41.0,5.924256,2.210684,6.7


## Pooled-quantile naive thresholds

Here `K` denotes the number of internal thresholds.

- `K = 2` uses pooled tertile cutoffs
- `K = 4` uses pooled quintile cutoffs

In [4]:
def pooled_quantile_thresholds(glucose_lists, threshold_count: int) -> tuple[np.ndarray, np.ndarray]:
    """Return pooled probability levels and pooled empirical quantile cutoffs."""
    pooled_glucose = np.concatenate([np.asarray(values, dtype=float) for values in glucose_lists])
    probs = np.arange(1, threshold_count + 1, dtype=float) / (threshold_count + 1)
    cutoffs = np.quantile(pooled_glucose, probs)
    if np.any(np.diff(cutoffs) <= 0):
        raise ValueError(f"Naive thresholds for K={threshold_count} are not strictly increasing: {cutoffs}")
    return probs, cutoffs


def tir_column_names(thresholds, minmax=(40, 401)) -> list[str]:
    """Recreate the TIR column names generated by `add_tir_metrics`."""
    extended = [minmax[0], *list(thresholds), minmax[1]]
    return [f"TIR_{int(np.ceil(extended[i]))}_{int(np.ceil(extended[i + 1] - 1))}" for i in range(len(extended) - 1)]

In [6]:
naive_threshold_specs = {
    "naive_k2": 2,
    "naive_k4": 4,
}

naive_threshold_map = {}
naive_threshold_rows = []

for label, threshold_count in naive_threshold_specs.items():
    probs, cutoffs = pooled_quantile_thresholds(filtered_data["gl"], threshold_count)
    naive_threshold_map[label] = cutoffs.tolist()
    naive_threshold_rows.append(
        {
            "Label": label,
            "Threshold count": threshold_count,
            "Bin count": threshold_count + 1,
            "Probability grid": ", ".join(f"{p:.3f}" for p in probs),
            "Cutoffs": ", ".join(f"{c:.1f}" for c in cutoffs),
        }
    )

naive_thresholds_df = pd.DataFrame(naive_threshold_rows).set_index("Label")
display(Markdown("#### Naive threshold summary"))
display(naive_thresholds_df)

#### Naive threshold summary

,Threshold count,Bin count,Probability grid,Cutoffs
Label,,,,
naive_k2,2,3,"0.333, 0.667","109.0, 129.0"
naive_k4,4,5,"0.200, 0.400, 0.600, 0.800","102.0, 113.0, 124.0, 141.0"


## Subject-level TIR summaries

The notebook now adds both the settled DE threshold features and the pooled-quantile naive threshold features to the same subject-level dataframe.

In [7]:
modeling_data = filtered_data.copy()
feature_map = {
    "de_k2": tir_column_names(DE_THRESHOLD_SETS["de_k2"]),
    "de_k4": tir_column_names(DE_THRESHOLD_SETS["de_k4"]),
}

for thresholds in DE_THRESHOLD_SETS.values():
    modeling_data = add_tir_metrics(modeling_data, thresholds=thresholds)

for label, thresholds in naive_threshold_map.items():
    modeling_data = add_tir_metrics(modeling_data, thresholds=thresholds)
    feature_map[label] = tir_column_names(thresholds)

modeling_data = modeling_data.loc[:, ~modeling_data.columns.duplicated()].copy()

feature_summary_df = pd.DataFrame(
    {
        "Predictor set": list(feature_map.keys()),
        "Columns used": [", ".join(cols[:-1]) for cols in feature_map.values()],
        "Dropped last bin": [cols[-1] for cols in feature_map.values()],
    }
).set_index("Predictor set")

display(Markdown("#### Predictor sets used in the linear models"))
display(feature_summary_df)

#### Predictor sets used in the linear models

,Columns used,Dropped last bin
Predictor set,,
de_k2,"TIR_40_95, TIR_96_169",TIR_170_400
de_k4,"TIR_40_89, TIR_90_127, TIR_128_171, TIR_172_231",TIR_232_400
naive_k2,"TIR_40_108, TIR_109_128",TIR_129_400
naive_k4,"TIR_40_101, TIR_102_112, TIR_113_123, TIR_124_140",TIR_141_400


## Linear-model comparisons

Required comparisons:

- naive `K = 2` versus DE `K = 2`
- naive `K = 4` versus DE `K = 4`

In [9]:
comparison_specs = [
    {
        "comparison": "K=2",
        "naive_key": "naive_k2",
        "de_key": "de_k2",
        "naive_label": "Naive K=2",
        "de_label": "DE K=2",
        "display_suffix": "(K=2)",
    },
    {
        "comparison": "K=4",
        "naive_key": "naive_k4",
        "de_key": "de_k4",
        "naive_label": "Naive K=4",
        "de_label": "DE K=4",
        "display_suffix": "(K=4)",
    },
]

naive_comparison_rows = []
naive_display_df = pd.DataFrame(index=[format_response_name(response) for response in RESPONSES])
model_store = {}

for response in RESPONSES:
    response_label = format_response_name(response)
    for spec in comparison_specs:
        naive_vars = feature_map[spec["naive_key"]][:-1]
        de_vars = feature_map[spec["de_key"]][:-1]

        naive_formula = f"{response} ~ " + " + ".join(naive_vars)
        de_formula = f"{response} ~ " + " + ".join(de_vars)

        naive_model = ols(naive_formula, data=modeling_data).fit()
        de_model = ols(de_formula, data=modeling_data).fit()

        packed_naive = pack_lm_model(naive_model)
        packed_de = pack_lm_model(de_model)
        clarke_stat, p_value = clarke_test(modeling_data, response, naive_vars, de_vars)
        delta_aic = packed_naive["aic"] - packed_de["aic"]

        naive_comparison_rows.append(
            {
                "Response": response_label,
                "Comparison": spec["comparison"],
                "Naive model": spec["naive_label"],
                "DE model": spec["de_label"],
                "R^2 naive": packed_naive["r2"],
                "R^2 DE": packed_de["r2"],
                "AIC naive": packed_naive["aic"],
                "AIC DE": packed_de["aic"],
                "AIC (Naive - DE)": delta_aic,
                "Clarke stat": clarke_stat,
                "p-value": p_value,
            }
        )

        suffix = spec["display_suffix"]
        naive_display_df.at[response_label, f"R^2 (Naive) {suffix}"] = f"{packed_naive['r2']:.3f}"
        naive_display_df.at[response_label, f"R^2 (DE) {suffix}"] = f"{packed_de['r2']:.3f}"
        naive_display_df.at[response_label, f"AIC (Naive - DE, p-value) {suffix}"] = f"{delta_aic:.1f} ({p_value:.3f})"

        model_store[(response, spec["comparison"])] = {
            "naive": naive_model,
            "de": de_model,
            "naive_vars": naive_vars,
            "de_vars": de_vars,
        }

naive_comparison_df = pd.DataFrame(naive_comparison_rows)
naive_numeric_df = naive_comparison_df.copy()

display(Markdown("#### Long-form comparison dataframe"))
display(naive_comparison_df)

display(Markdown("#### Final display table"))
display(naive_display_df)

#### Long-form comparison dataframe

,Response,Comparison,Naive model,DE model,R^2 naive,R^2 DE,AIC naive,AIC DE,AIC (Naive - DE),Clarke stat,p-value
0,HDL-C,K=2,Naive K=2,DE K=2,0.035426,0.017898,4829.923354,4840.241792,-10.318438,343,0.000003
1,HDL-C,K=4,Naive K=4,DE K=4,0.039545,0.041118,4831.470968,4830.531586,0.939382,276,0.403450
2,TG,K=2,Naive K=2,DE K=2,0.050388,0.049036,882.889971,883.705407,-0.815436,289,0.867308
3,TG,K=4,Naive K=4,DE K=4,0.052783,0.052314,885.442988,885.726539,-0.283551,297,0.403450
4,TG/HDL-C,K=2,Naive K=2,DE K=2,0.059407,0.050413,1169.723183,1175.176242,-5.453059,318,0.009535
5,TG/HDL-C,K=4,Naive K=4,DE K=4,0.061651,0.059510,1172.354871,1173.660789,-1.305918,323,0.002601


#### Final display table

,R^2 (Naive) (K=2),R^2 (DE) (K=2),"AIC (Naive - DE, p-value) (K=2)",R^2 (Naive) (K=4),R^2 (DE) (K=4),"AIC (Naive - DE, p-value) (K=4)"
HDL-C,0.035,0.018,-10.3 (0.000),0.040,0.041,0.9 (0.403)
TG,0.050,0.049,-0.8 (0.867),0.053,0.052,-0.3 (0.403)
TG/HDL-C,0.059,0.050,-5.5 (0.010),0.062,0.060,-1.3 (0.003)


In [11]:
summary_lines = [
    "#### Quick reading guide",
    "",
    "Positive values in `AIC (Naive - DE)` favor the DE model because they indicate a smaller AIC for DE.",
    "",
]

for _, row in naive_comparison_df.iterrows():
    favored_r2 = row["DE model"] if row["R^2 DE"] > row["R^2 naive"] else row["Naive model"]
    favored_aic = row["DE model"] if row["AIC (Naive - DE)"] > 0 else row["Naive model"]
    summary_lines.append(
        f"- **{row['Response']} / {row['Comparison']}**: higher R^2 = {favored_r2}; lower AIC = {favored_aic}; Clarke p-value = {row['p-value']:.3f}."
    )

display(Markdown("\n".join(summary_lines)))

#### Quick reading guide

Positive values in `AIC (Naive - DE)` favor the DE model because they indicate a smaller AIC for DE.

- **HDL-C / K=2**: higher R^2 = Naive K=2; lower AIC = Naive K=2; Clarke p-value = 0.000.
- **HDL-C / K=4**: higher R^2 = DE K=4; lower AIC = DE K=4; Clarke p-value = 0.403.
- **TG / K=2**: higher R^2 = Naive K=2; lower AIC = Naive K=2; Clarke p-value = 0.867.
- **TG / K=4**: higher R^2 = Naive K=4; lower AIC = Naive K=4; Clarke p-value = 0.403.
- **TG/HDL-C / K=2**: higher R^2 = Naive K=2; lower AIC = Naive K=2; Clarke p-value = 0.010.
- **TG/HDL-C / K=4**: higher R^2 = Naive K=4; lower AIC = Naive K=4; Clarke p-value = 0.003.